# Projeto Integrador - AeroMotion Dynamics
## Etapa 2: Modelagem Preditiva Multiclasse, Detecção de Anomalias e MLOps

Treinamento comparativo de classificadores supervisionados, balanceamento com SMOTE, detecção de desvios operacionais com Isolation Forest e exportação de artefatos.

In [ ]:
import os
from pathlib import Path
import sys

# Ajusta o diretório ativo para a raiz do repositório
caminho_raiz = Path.cwd().parent
if str(caminho_raiz) not in sys.path:
  sys.path.append(str(caminho_raiz))
os.chdir(caminho_raiz)

import pandas as pd
from src.ml.evaluation import IndustrialModelEvaluator
from src.ml.train import IndustrialModelTrainer

print(f"Diretório de trabalho ativo: {os.getcwd()}")

## 1. Divisão Temporal Estrita (70% Treino / 30% Teste) e Normalização
Os dados são particionados cronologicamente sem aleatorização (*shuffle=False*) para evitar contaminação de tendências futuras no conjunto de treino. O `RobustScaler` é ajustado apenas sobre o treino.

In [ ]:
trainer = IndustrialModelTrainer(
    "data/processed/features_eng.parquet"
)
X_train, X_test, y_train, y_test = trainer.preparar_dados(
    target_col="tipo_falha", proporcao_treino=0.7
)

print(f"Dimensões do Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Distribuição do Target no Treino Bruto:\n{y_train.value_counts()}")

# Aplicação de SMOTE exclusivamente no treino
X_train_smote, y_train_smote = trainer.balancear_treino_smote()
print(
    f"\nDistribuição do Target no Treino com SMOTE:\n{y_train_smote.value_counts()}"
)

## 2. Modelagem Multiclasse Supervisionada
Treinamento e comparação entre `RandomForestClassifier` e `LGBMClassifier` com busca de hiperparâmetros focada na maximização do *Recall Macro*.

In [ ]:
modelos_treinados = trainer.treinar_supervisionados(
    X_train_smote, y_train_smote
)

relatorios = {}
for nome, modelo in modelos_treinados.items():
  relatorios[nome] = IndustrialModelEvaluator.avaliar_classificador(
      modelo, X_test, y_test, nome
  )

# Tabela comparativa de métricas
tabela_comparativa = IndustrialModelEvaluator.comparar_desempenho(
    relatorios
)
display(tabela_comparativa)

## 3. Detecção Não Supervisionada de Anomalias (Isolation Forest)
Treinamento de um estimador `Isolation Forest` treinado exclusivamente com amostras nominais (`tipo_falha == 0`) para sinalizar assinaturas de degradação não mapeadas previamente.

In [ ]:
modelo_iso = trainer.treinar_isolation_forest()

# Verificação de anomalias no conjunto de teste (-1 indica anomalia, 1 indica normalidade)
pred_iso = modelo_iso.predict(X_test)
total_anomalias = (pred_iso == -1).sum()
print(
    f"Total de registros sinalizados como anomalia no conjunto de teste: {total_anomalias} ({total_anomalias / len(X_test) * 100:.2f}%)"
)

## 4. Explicabilidade Física e Persistência de Modelos
Identificação das variáveis com maior impacto na classificação de modos de falha e serialização dos modelos (`.joblib`) no diretório `models/`.

In [ ]:
# Seleção do modelo com melhor Recall Macro
melhor_nome = tabela_comparativa.sort_values(
    by="Recall Macro", ascending=False
).iloc[0]["Modelo"]
melhor_modelo = modelos_treinados[melhor_nome]
print(f"Modelo selecionado para produção: {melhor_nome}")

# Explicabilidade física
IndustrialModelEvaluator.plotar_feature_importance(
    melhor_modelo, trainer.feature_names, top_n=10
)

# Exportação dos artefatos
trainer.salvar_artefactos(
    melhor_classificador=melhor_modelo,
    modelo_iso=modelo_iso,
    pasta_destino="models",
)